<a href="https://colab.research.google.com/github/NxrFesdac/bourbaki-nlp-avanzado/blob/main/modulo5-bbva/memento.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U langchain-openai langchain-community langgraph faiss-cpu openai duckdb pydantic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.8/108.8 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.8/173.8 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 69.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.4/21.4 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.0/472.0 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 

In [ ]:
import duckdb
import os
import getpass
import time
import torch
import torch.nn as nn
import torch.optim as optim
from typing import TypedDict, List
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, END

# ==========================================
# 1. SETUP & API
# ==========================================
os.environ["OPENAI_API_KEY"] = getpass.getpass("Introduce tu OpenAI API Key: ")

model = ChatOpenAI(model="gpt-4.1-mini", temperature=0.1)
embeddings = OpenAIEmbeddings()

# ==========================================
# 2. DATABASE
# ==========================================
con = duckdb.connect(database=':memory:')
setup_sql = """
-- USERS
CREATE TABLE users (id INT PRIMARY KEY, first_name VARCHAR(50), last_name VARCHAR(50), email VARCHAR(100) UNIQUE, created_at DATE);
INSERT INTO users VALUES
(1, 'Ana', 'Gomez', 'ana@example.com', '2023-01-15'),
(2, 'Carlos', 'Ruiz', 'carlos@example.com', '2023-03-20'),
(3, 'Belen', 'Soto', 'belen@example.com', '2023-05-10');

-- CATEGORIES
CREATE TABLE categories (id INT PRIMARY KEY, name VARCHAR(100));
INSERT INTO categories VALUES (1, 'Electronics'), (2, 'Apparel'), (3, 'Home');

-- PRODUCTS
CREATE TABLE products (id INT PRIMARY KEY, category_id INT, name VARCHAR(255), price DECIMAL(10, 2), stock_quantity INT, is_active BOOLEAN);
INSERT INTO products VALUES
(1, 1, 'Pro Laptop 15', 1299.99, 15, TRUE),
(2, 1, 'Wireless Mouse', 49.99, 50, TRUE),
(3, 2, 'Winter Coat', 199.99, 10, FALSE), -- Inventario Muerto
(4, 2, 'Summer Dress', 39.99, 0, FALSE),  -- No es inventario muerto (stock 0)
(5, 3, 'Table Lamp', 35.50, 20, FALSE),   -- Inventario Muerto
(6, 3, 'Desk Chair', 150.00, 5, TRUE);

-- ORDERS
CREATE TABLE orders (id INT PRIMARY KEY, user_id INT, total_amount DECIMAL(10, 2), status VARCHAR(20));
INSERT INTO orders VALUES
(101, 1, 1349.98, 'DELIVERED'),
(102, 2, 39.99, 'DELIVERED'),
(103, 1, 150.00, 'DELIVERED'),
(104, 3, 2500.00, 'DELIVERED'), -- Belen es VIP (>2000)
(105, 3, 100.00, 'SHIPPED');
"""
con.execute(setup_sql)

# Vector DB for Schemas
table_schemas_docs = [
    Document(page_content="Tabla: categories\nEsquema: id INT, name VARCHAR", metadata={"table_name": "categories"}),
    Document(page_content="Tabla: users\nEsquema: id INT, first_name VARCHAR, last_name VARCHAR, email VARCHAR, created_at DATE", metadata={"table_name": "users"}),
    Document(page_content="Tabla: products\nEsquema: id INT, category_id INT, name VARCHAR, price DECIMAL, stock_quantity INT, is_active BOOLEAN", metadata={"table_name": "products"}),
    Document(page_content="Tabla: orders\nEsquema: id INT, user_id INT, total_amount DECIMAL, status VARCHAR", metadata={"table_name": "orders"}),
]
schema_db = FAISS.from_documents(table_schemas_docs, embeddings)
schema_retriever = schema_db.as_retriever(search_kwargs={"k": 3})

# ==========================================
# 3. GLOBAL TOOLS
# ==========================================
@tool
def search_schema(query: str) -> str:
    """Busca esquemas de tablas SQL relevantes."""
    docs = schema_retriever.invoke(query)
    return "\n\n".join([f"Tabla: {d.metadata['table_name']}\n{d.page_content}" for d in docs])

@tool
def execute_sql(query: str) -> str:
    """Ejecuta una consulta SQL en DuckDB."""
    try:
        result_df = con.execute(query).df()
        return result_df.to_markdown(index=False) if not result_df.empty else "0 filas devueltas."
    except Exception as e:
        return f"Error SQL: {e}"

# Shared executor tool
base_executor = create_react_agent(model=model, tools=[search_schema, execute_sql])

Introduce tu OpenAI API Key: ··········


/tmp/ipykernel_1008/3498241258.py:91: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  base_executor = create_react_agent(model=model, tools=[search_schema, execute_sql])


In [ ]:
# ==========================================
# FASE A: EL AGENTE ÚNICO
# ==========================================
print("\n" + "="*60)
print("🔴 FASE A: EL AGENTE ÚNICO (Sin planificación, sin memoria)")
print("="*60)

system_prompt_single = "Eres un experto en SQL. Crea queries basado en lo que se te solicita"
single_agent = create_react_agent(model=model, tools=[search_schema, execute_sql], prompt=system_prompt_single)

query_jargon_1 = "Calcula el valor de nuestro 'Inventario Muerto'."
print(f"🗣️ [Usuario]: {query_jargon_1}")

for event in single_agent.stream({"messages": [("user", query_jargon_1)]}, stream_mode="values"):
    if "messages" in event:
        event["messages"][-1].pretty_print()

print("\n❌ EXPLICACIÓN: El agente no entiende la regla de un inventario muerto.")
time.sleep(2)

/tmp/ipykernel_1008/160667193.py:9: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  single_agent = create_react_agent(model=model, tools=[search_schema, execute_sql], prompt=system_prompt_single)



🔴 FASE A: EL AGENTE ÚNICO (Sin planificación, sin memoria)
🗣️ [Usuario]: Calcula el valor de nuestro 'Inventario Muerto'.
================================ Human Message =================================

Calcula el valor de nuestro 'Inventario Muerto'.
================================== Ai Message ==================================
Tool Calls:
  search_schema (call_tWrgm36Lu5oeYItNlr8WzWNU)
 Call ID: call_tWrgm36Lu5oeYItNlr8WzWNU
  Args:
    query: inventario muerto
================================= Tool Message =================================
Name: search_schema

Tabla: products
Tabla: products
Esquema: id INT, category_id INT, name VARCHAR, price DECIMAL, stock_quantity INT, is_active BOOLEAN

Tabla: categories
Tabla: categories
Esquema: id INT, name VARCHAR

Tabla: users
Tabla: users
Esquema: id INT, first_name VARCHAR, last_name VARCHAR, email VARCHAR, created_at DATE
================================== Ai Message ==================================

Para calcular el valor del 'In

In [ ]:
# ==========================================
# FASE B: PLANIFICADOR + EJECUTOR
# ==========================================
print("\n" + "="*60)
print("🟡 FASE B: PLANIFICADOR + EJECUTOR (Sin memoria)")
print("="*60)

class BaseAgentState(TypedDict):
    user_query: str
    plan: str
    final_answer: str

def basic_executor_node(state: BaseAgentState):
    prompt = f"Resuelve: '{state['user_query']}'.\nSIGUE ESTE PLAN:\n{state['plan']}"
    result = base_executor.invoke({"messages": [HumanMessage(content=prompt)]})
    return {"final_answer": result["messages"][-1].content}

def basic_planner_node(state: BaseAgentState):
    prompt = "Eres el Planificador. Crea un plan paso a paso para que el ejecutor escriba SQL. No uses herramientas."
    response = model.invoke([SystemMessage(content=prompt), HumanMessage(content=state['user_query'])])
    return {"plan": response.content}

basic_workflow = StateGraph(BaseAgentState)
basic_workflow.add_node("planner", basic_planner_node)
basic_workflow.add_node("executor", basic_executor_node)
basic_workflow.set_entry_point("planner")
basic_workflow.add_edge("planner", "executor")
basic_workflow.add_edge("executor", END)
basic_app = basic_workflow.compile()

query_jargon_2 = "¿Quiénes son nuestros 'Clientes VIP' y cuánto han gastado en total?"
print(f"🗣️ [Usuario]: {query_jargon_2}")

res = basic_app.invoke({"user_query": query_jargon_2})
print(f"\n📝 PLAN GENERADO:\n{res['plan']}\n\n⚙️ RESPUESTA FINAL:\n{res['final_answer']}")

print("\n❌ EXPLICACIÓN: Divide bien la tarea, pero inventa la regla de los 'Clientes VIP'. Sigue fallando.")
time.sleep(2)


🟡 FASE B: PLANIFICADOR + EJECUTOR (Sin memoria)
🗣️ [Usuario]: ¿Quiénes son nuestros 'Clientes VIP' y cuánto han gastado en total?

📝 PLAN GENERADO:
Para responder a la pregunta "¿Quiénes son nuestros 'Clientes VIP' y cuánto han gastado en total?", primero necesitamos definir qué significa "Clientes VIP" en el contexto de la base de datos. Supongamos que "Clientes VIP" son aquellos que han gastado por encima de cierto umbral en total.

Aquí tienes un plan paso a paso para escribir la consulta SQL:

1. **Identificar las tablas relevantes**:
   - Tabla de clientes (por ejemplo, `clientes`) que contiene información de los clientes.
   - Tabla de ventas o pedidos (por ejemplo, `ventas` o `pedidos`) que contiene los registros de compras realizadas por los clientes.

2. **Determinar las columnas necesarias**:
   - En la tabla de clientes: `cliente_id`, `nombre` (u otra columna que identifique al cliente).
   - En la tabla de ventas: `cliente_id`, `monto` (o `total`), que indica cuánto gastó 

In [ ]:
# ==========================================
# FASE C: MEMENTO SIMPLE (FAISS + Humano)
# ==========================================
print("\n" + "="*60)
print("🔵 FASE C: MEMENTO SIMPLE (Memoria Episódica Básica + Feedback Humano)")
print("="*60)

class SimpleArchivist:
    def __init__(self):
        self.db = FAISS.from_texts(["init"], embeddings) #documento dummy para arrancar vectorDB
        self.db.delete([self.db.index_to_docstore_id[0]]) #removemos el documento dummy que cargamos
        self.count = 0

    def save_case(self, task: str, plan: str, success: bool, feedback: str):
        status = "SUCCESS" if success else "FAILURE"
        content = f"[RESULTADO: {status}]\nTarea: {task}\nPlan Usado: {plan}\nCorrección/Regla: {feedback}"
        doc = Document(page_content=content, metadata={"task": task})
        self.db.add_documents([doc])
        self.count += 1

    def retrieve(self, task: str, k: int = 1) -> str:
        if self.count == 0: return "No hay casos previos."
        docs = self.db.similarity_search(task, k=k)
        return "\n\n".join([d.page_content for d in docs])

simple_archivist = SimpleArchivist()

def simple_memento_planner_node(state: BaseAgentState):
    past_cases = simple_archivist.retrieve(state['user_query'])
    prompt = f"""Eres un Planificador con Memoria Simple. Crea un plan para el ejecutor de SQL.
    === CASOS PASADOS (FAISS Similarity) ===
    {past_cases}
    ===============================
    Instrucciones: Lee atentamente la 'Corrección/Regla' e inyecta esa lógica en tu nuevo plan."""
    response = model.invoke([SystemMessage(content=prompt), HumanMessage(content=state['user_query'])])
    return {"plan": response.content}

simple_memento_workflow = StateGraph(BaseAgentState)
simple_memento_workflow.add_node("planner", simple_memento_planner_node)
simple_memento_workflow.add_node("executor", basic_executor_node)
simple_memento_workflow.set_entry_point("planner")
simple_memento_workflow.add_edge("planner", "executor")
simple_memento_workflow.add_edge("executor", END)
simple_memento_app = simple_memento_workflow.compile()

def run_simple_memento(query: str, mock_success=None, mock_feedback=None):
    print(f"\n🗣️ [Usuario]: {query}")
    final_plan = ""
    for event in simple_memento_app.stream({"user_query": query}):
        if "planner" in event:
            final_plan = event["planner"]["plan"]
            print(f"📝 [Planificador Simple]: Plan generado y pasado al ejecutor.")
        elif "executor" in event:
            print(f"⚙️ [Ejecutor SQL]:\n{event['executor']['final_answer']}")

    if mock_feedback:
        print(f"\n👨‍💻 [Feedback Humano Simulado]: Éxito: {mock_success} | Regla: {mock_feedback}")
        simple_archivist.save_case(query, final_plan, mock_success, mock_feedback)
        print("💾 Memoria guardada en FAISS.")

print("\n--- INTENTO 1 (FAISS Vacío) ---")
run_simple_memento("Calcula el valor del Inventario Muerto.",
                   mock_success=False,
                   mock_feedback="El Inventario Muerto son productos donde stock_quantity > 0 AND is_active = FALSE. Multiplica price * stock_quantity.")

time.sleep(2)
print("\n--- INTENTO 2 (FAISS con Memoria) ---")
run_simple_memento("¿Cuánto valor tenemos en Inventario Muerto ahora mismo?")

print("\n⚠️ EXPLICACIÓN: Funciona, pero depende de retroalimentación humana constante y la búsqueda FAISS básica puede recuperar casos irrelevantes si la base de datos crece mucho.")
time.sleep(2)


🔵 FASE C: MEMENTO SIMPLE (Memoria Episódica Básica + Feedback Humano)

--- INTENTO 1 (FAISS Vacío) ---

🗣️ [Usuario]: Calcula el valor del Inventario Muerto.
📝 [Planificador Simple]: Plan generado y pasado al ejecutor.
⚙️ [Ejecutor SQL]:
En la base de datos, la tabla que parece estar relacionada con el inventario es la tabla "products". Los campos relevantes para el inventario podrían ser:

- id (identificador del producto)
- category_id (categoría del producto)
- name (nombre del producto)
- price (precio unitario del producto)
- stock_quantity (cantidad en inventario)
- is_active (indica si el producto está activo)

Para identificar el Inventario Muerto, necesitaría saber cómo se define en tu contexto. Por ejemplo, podría ser productos que no están activos (is_active = false), o productos que no se han vendido en un tiempo determinado (aunque no tenemos información de ventas en las tablas mostradas).

¿Podrías confirmar cómo se define el Inventario Muerto en tu caso? ¿O si hay algun

In [ ]:
# ============================================================
# 🟢 FASE D: TRUE MEMENTO (Q-Learning + Subtasks + Evaluación Automática)
# ============================================================
print("\n" + "="*60)
print("🟢 FASE D: TRUE MEMENTO (Q-Learning + Subtasks + Evaluación Automática)")
print("="*60)

class QNetwork(nn.Module):
    def __init__(self, embed_dim=1536):
        super().__init__()
        self.fc1 = nn.Linear(embed_dim * 2, 512) # embedding de pregunta actual + planes anteriores
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(512, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, state_emb, case_emb):
        x = torch.cat([state_emb, case_emb], dim=-1)
        x = self.relu(self.fc1(x))
        return self.sigmoid(self.fc2(x))

class ParametricArchivist:
    def __init__(self, embedder):
        self.embedder = embedder
        self.q_net = QNetwork()
        self.optimizer = optim.Adam(self.q_net.parameters(), lr=0.01)
        self.criterion = nn.BCELoss()
        self.case_bank = []

    def embed(self, text: str) -> torch.Tensor:
        vector = self.embedder.embed_query(text)
        return torch.tensor(vector, dtype=torch.float32)

    def retrieve_top_k(self, current_query: str, k: int = 2) -> str:
        if not self.case_bank: return "No hay casos previos en la memoria."
        state_emb = self.embed(current_query)
        scored_cases = []

        self.q_net.eval()
        with torch.no_grad():
            for case in self.case_bank:
                case_emb = self.embed(f"Task: {case['query']}\nPlan: {case['plan']}")
                q_value = self.q_net(state_emb, case_emb).item() # Concatenación del embedding de la pregunta actual + planes anteriores
                scored_cases.append((q_value, case))

        scored_cases.sort(key=lambda x: x[0], reverse=True)
        top_k = scored_cases[:k]

        print("\n🔍 [Motor de Recuperación Q-Learning]:")
        for score, c in top_k:
            print(f"   -> Q-Value Estimado: {score:.4f} | Tarea: '{c['query']}'")

        return "\n\n".join([f"[Q-Score: {s:.4f} | Reward Pasado: {c['reward']}]\nTarea: {c['query']}\nPlan: {c['plan']}\nFEEDBACK: {c.get('feedback', 'N/A')}" for s, c in top_k])

    def save_and_train(self, query: str, plan: str, reward: float, feedback: str):
        self.case_bank.append({'query': query, 'plan': plan, 'reward': reward, 'feedback': feedback})
        self.q_net.train()
        self.optimizer.zero_grad()

        state_emb = self.embed(query)
        case_emb = self.embed(f"Task: {query}\nPlan: {plan}")

        q_pred = self.q_net(state_emb, case_emb).squeeze()
        target_val = 1.0 if reward >= 0.5 else 0.0
        target = torch.tensor(target_val, dtype=torch.float32)

        loss = self.criterion(q_pred, target)
        loss.backward()
        self.optimizer.step()
        print(f"🧠 [Red Q Actualizada] Predicción Inicial: {q_pred.item():.4f} | Pérdida (Loss): {loss.item():.4f}")

parametric_archivist = ParametricArchivist(embeddings)

class MementoState(TypedDict):
    user_query: str; case_memory: str; subtasks: List[str]; final_answer: str; reward: float

def true_planner_node(state: MementoState):
    past_cases = parametric_archivist.retrieve_top_k(state['user_query'])

    # PROMPT MEJORADO: Forzamos al planificador a traducir el feedback en reglas SQL literales.
    prompt = f"""Eres el Planificador Memento. Descompón la tarea en pasos.
    === CASOS PREVIOS (Q-Valued) ===
    {past_cases}
    ================================
    INSTRUCCIÓN VITAL:
    1. Revisa el FEEDBACK de los casos previos.
    2. El Ejecutor SQL NO tiene acceso a esta memoria. Por lo tanto, DEBES escribir las reglas matemáticas y filtros exactos en tu plan.
    Ejemplo Malo: "Paso 1: Filtrar Clientes VIP".
    Ejemplo Bueno: "Paso 1: Filtrar la tabla orders agrupando por usuario donde SUM(total_amount) > 2000".

    Devuelve UNA LISTA NUMERADA de pasos explícitos."""

    response = model.invoke([SystemMessage(content=prompt), HumanMessage(content=state['user_query'])])
    return {"subtasks": [step for step in response.content.split('\n') if step.strip()], "case_memory": past_cases}

def memento_executor_node(state: MementoState):
    plan_str = "\n".join(state['subtasks'])
    result = base_executor.invoke({"messages": [HumanMessage(content=f"Resuelve: '{state['user_query']}'.\nSIGUE ESTE PLAN ESTRICTAMENTE:\n{plan_str}")]})
    return {"final_answer": result["messages"][-1].content}

def automated_evaluator_node(state: MementoState):
    # PROMPT CORREGIDO: Forzamos al evaluador a no verificar los datos, sino la lógica.
    prompt = f"""Eres un Evaluador Automático Estricto.
    Pregunta Original: {state['user_query']}
    Respuesta Generada: {state['final_answer']}
    Pasos Usados: {state['subtasks']}

    GLOSARIO ESTRICTO DE LA EMPRESA:
    - "Inventario Muerto": Productos donde stock_quantity > 0 AND is_active = FALSE.
    - "Cliente VIP": Usuarios cuya suma de total_amount en orders es > 2000.

    REGLAS DE EVALUACIÓN VITALES (LEE ATENTAMENTE):
    1. ASUME QUE LOS DATOS SON 100% CORRECTOS: No tienes acceso a la base de datos. Confía en que los nombres (ej. Belén, Ana, Winter Coat) y los números devueltos en la 'Respuesta Generada' son reales. NO castigues a la respuesta por los datos.
    2. EVALÚA SOLO LA LÓGICA: Tu único trabajo es revisar los 'Pasos Usados'. Si los pasos incluyen los filtros matemáticos y lógicos exactos del glosario (ej. "stock_quantity > 0 AND is_active = FALSE" o "> 2000"), el REWARD DEBE SER 1.0.
    3. MOTIVOS PARA FALLAR (REWARD 0.0): Si los pasos asumen reglas distintas (ej. asumir que muerto es stock = 0), si la respuesta pide aclaraciones al usuario, o si no aplicó el glosario.

    Responde EXACTAMENTE en dos líneas:
    REWARD: [0.0 o 1.0]
    FEEDBACK: [Explicación concisa y directa citando la regla del glosario. Si es 1.0, felicita la lógica usada en los pasos.]"""

    texto = model.invoke([HumanMessage(content=prompt)]).content.strip()
    reward, feedback = 0.0, "Sin feedback"

    for linea in texto.split('\n'):
        if linea.startswith("REWARD:"):
            try: reward = float(linea.replace("REWARD:", "").strip())
            except: pass
        elif linea.startswith("FEEDBACK:"): feedback = linea.replace("FEEDBACK:", "").strip()

    print(f"⚖️ [Auto-Evaluador] Reward: {reward} | Feedback: {feedback}")
    parametric_archivist.save_and_train(state['user_query'], "\n".join(state['subtasks']), reward, feedback)
    return {"reward": reward}

m_mdp_workflow = StateGraph(MementoState)
m_mdp_workflow.add_node("planner", true_planner_node)
m_mdp_workflow.add_node("executor", memento_executor_node)
m_mdp_workflow.add_node("evaluator", automated_evaluator_node)
m_mdp_workflow.set_entry_point("planner")
m_mdp_workflow.add_edge("planner", "executor")
m_mdp_workflow.add_edge("executor", "evaluator")
m_mdp_workflow.add_edge("evaluator", END)
true_memento_app = m_mdp_workflow.compile()

def run_true_memento(query: str):
    print(f"\n🗣️ [Usuario]: {query}")
    for event in true_memento_app.stream({"user_query": query}):
        if "planner" in event:
            print("\n📝 [Planificador] Plan Generado:")
            for step in event["planner"]["subtasks"]: print(f"   {step}")
        if "executor" in event:
            print(f"\n⚙️ [Ejecutor SQL] Resultado Final:\n{event['executor']['final_answer']}")

# ==========================================
# SECUENCIA DE PRUEBA (4 ITERACIONES)
# ==========================================

print("\n--- INTENTO 1 Inventario Muerto ---")
run_true_memento("Calcula el valor financiero del Inventario Muerto.")

time.sleep(2)
print("\n--- INTENTO 2 Inventario Muerto ---")
run_true_memento("Haz una lista con los nombres de los productos que son Inventario Muerto.")

time.sleep(2)
print("\n--- INTENTO 3 Clientes VIP ---")
run_true_memento("¿Quiénes son nuestros Clientes VIP y cuánto gastaron?")

time.sleep(2)
print("\n--- INTENTO 4 Clientes VIP ---")
run_true_memento("Dime los correos de nuestros Clientes VIP.")


🟢 FASE D: TRUE MEMENTO (Q-Learning + Subtasks + Evaluación Automática)

--- INTENTO 1 Inventario Muerto ---

🗣️ [Usuario]: Calcula el valor financiero del Inventario Muerto.

📝 [Planificador] Plan Generado:
   Para calcular el valor financiero del Inventario Muerto, sigue estos pasos detallados:
   1. Identificar los productos en inventario que no han tenido movimientos de venta ni salida en un período determinado (por ejemplo, 6 meses o 1 año).  
      - Consulta la tabla de inventario y la tabla de movimientos o ventas.  
      - Filtra los productos cuyo último movimiento de salida o venta sea anterior a la fecha límite (fecha actual menos el período definido).  
   2. Obtener la cantidad disponible en inventario para esos productos identificados.  
      - Consulta la columna de cantidad en la tabla de inventario para esos productos.  
   3. Obtener el costo unitario de cada producto en inventario.  
      - Consulta la columna de costo unitario en la tabla de inventario o en la t